# 3 · ai_classify y Jev: pedir una decisión con el mismo criterio

Comparamos la función de Databricks `ai_classify` con Jev sobre los **mismos ocho títulos como máximo** y la misma taxonomía. Si el poll C2 aún no tiene títulos, el notebook declara que usa el pequeño conjunto sintético de enseñanza. Las respuestas de participantes se procesan de forma acotada y nunca aparecen en una salida; el cuadro enseña solo clase, confianza y latencia.

Jev usa el secreto de Databricks `ai4data-class2 / typesafe-api-key`. Para crear o reemplazarlo desde `.env`, ejecuta `uv run demos/clase-02/notebooks/setup_secret.py`. El notebook nunca imprime ni persiste el valor.

La coincidencia entre sistemas no es exactitud. Sin etiquetas humanas independientes solo medimos acuerdo, cobertura, confianza y tiempo.

In [ ]:
import json
import time
import urllib.error
import urllib.request
import pandas as pd
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "ai4data"
DIM = f"{CATALOG}.{SCHEMA}.dim_participante"
MAX_SAMPLE = 8
PERSONA_LABELS = ["ejecutivo", "manager", "practitioner", "estudiante", "otro"]
PERSONA_CRITERIA = {
    "ejecutivo": "C-level, director/a, VP, jefe/a de área",
    "manager": "lidera un equipo; gerente, product manager",
    "practitioner": "analista, ingeniero/a, científico/a de datos que ejecuta trabajo técnico",
    "estudiante": "estudia, becario/a, en formación",
    "otro": "no es un puesto o no encaja en las anteriores",
}

synthetic_titles = [
    "VP de Data", "Gerente de producto digital", "Ingeniera de datos",
    "Estudiante de estadística", "Diseñador gráfico", "Jefa de analítica",
    "Machine Learning Engineer", "Médico internista",
]
titles = []
try:
    catalog, schema, table = DIM.split(".")
    available = {r.tableName for r in spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect()}
    if table in available:
        titles = [
            r.puesto_texto for r in (
                spark.table(DIM)
                .where(F.col("puesto_texto").isNotNull() & (F.length(F.trim("puesto_texto")) > 0))
                .orderBy("participant_key")
                .select("puesto_texto")
                .limit(MAX_SAMPLE)
                .collect()
            )
        ]
except Exception as exc:
    if "TABLE_OR_VIEW_NOT_FOUND" not in str(exc) and "SCHEMA_NOT_FOUND" not in str(exc):
        raise

if titles:
    SAMPLE_SOURCE = "C2 · respuestas en vivo (muestra acotada)"
else:
    titles = synthetic_titles[:MAX_SAMPLE]
    SAMPLE_SOURCE = "fixture sintético · el poll C2 sigue vacío o no se ha cargado"
print(f"Fuente: {SAMPLE_SOURCE}; títulos procesados: {len(titles)}")

## Primera decisión: `ai_classify`

Ejecutamos una sola consulta sobre la muestra. No seleccionamos el texto al resultado visible.

In [ ]:
inputs = spark.createDataFrame(
    [(i + 1, title) for i, title in enumerate(titles)],
    ["muestra", "puesto_texto"]
)
labels_sql = ", ".join("'" + label.replace("'", "''") + "'" for label in PERSONA_LABELS)
ai_rows = (
    inputs.select(
        "muestra",
        F.expr(f"ai_classify(puesto_texto, array({labels_sql}))").alias("ai_classify"),
    )
    .orderBy("muestra")
    .collect()
)
ai_by_sample = {int(r.muestra): r.ai_classify for r in ai_rows}
display(pd.DataFrame([
    {"muestra": i, "ai_classify": ai_by_sample.get(i)}
    for i in range(1, len(titles) + 1)
]))

## Segunda decisión: Jev

La llamada está limitada a ocho títulos, tiene timeout y hasta dos reintentos. El secreto no entra en la celda de salida.

In [ ]:
API_URL = "https://api.typesafe.ai/v1/systemone"
API_KEY = dbutils.secrets.get(scope="ai4data-class2", key="typesafe-api-key")

def ask_jev(title):
    payload = {
        "state": f"Puesto: {title}",
        "model": "jev-latest",
        "questions": {
            "persona": {
                "type": "choice",
                "instructions": "¿Qué perfil describe mejor este puesto?",
                "criteria": PERSONA_CRITERIA,
            }
        },
    }
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    last_error = None
    for attempt in range(2):
        req = urllib.request.Request(API_URL, data=body, headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        }, method="POST")
        started = time.perf_counter()
        try:
            with urllib.request.urlopen(req, timeout=30) as response:
                result = json.loads(response.read())
            answer = result.get("answers", {}).get("persona", {})
            return {
                "persona_jev": answer.get("choice"),
                "confianza_jev": answer.get("confidence"),
                "latencia_jev_ms": round((time.perf_counter() - started) * 1000),
                "modelo_jev": result.get("model"),
            }
        except (urllib.error.URLError, TimeoutError) as exc:
            last_error = exc
            if attempt == 0:
                time.sleep(1)
    raise RuntimeError(f"Jev falló después de dos intentos: {last_error}")

jev_rows = [ask_jev(title) for title in titles]

## Comparación de resultados

Mostramos solo etiquetas y medidas operativas: ningún título ni clave de participante sale en la tabla.

In [ ]:
comparison = pd.DataFrame([
    {
        "muestra": i,
        "ai_classify": ai_by_sample.get(i),
        "Jev": jev_rows[i - 1]["persona_jev"],
        "confianza Jev": jev_rows[i - 1]["confianza_jev"],
        "latencia Jev (ms)": jev_rows[i - 1]["latencia_jev_ms"],
    }
    for i in range(1, len(titles) + 1)
])
comparison["acuerdo"] = comparison["ai_classify"] == comparison["Jev"]
display(comparison)
agreement = comparison["acuerdo"].mean() if len(comparison) else float("nan")
print(f"Acuerdo entre métodos: {agreement:.0%}. Esto no mide exactitud.")
print("Para medir exactitud necesitamos títulos con etiquetas independientes, representativos de la población y revisados por personas.")
dbutils.notebook.exit(
    f"AI_JEV_COMPLETE|source={SAMPLE_SOURCE}|n={len(comparison)}|agreement={agreement:.3f}"
)